In [40]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from pathlib import Path

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

from sklearn.preprocessing import LabelEncoder


In [41]:
dir = Path('/kaggle/input/nfl-big-data-bowl-2026-prediction')
trial = True


def read():
    train_in= pd.read_csv(dir / 'train/input_2023_w01.csv')
    train_out = pd.read_csv(dir / 'train/output_2023_w01.csv')
    test_in= pd.read_csv(dir / 'test_input.csv')
    test_out = pd.read_csv(dir / 'test.csv')
    
    if trial:
        mask = (train_in['player_to_predict']==True) | (train_in['game_id']==2023090700)
        train_in = train_in[mask]
        return train_in, train_out, test_in, test_out
    for i in range(2, 19):
        train_in = pd.concat([train_in, pd.read_csv(dir / 'train/input_2023_w{}.csv'.format('0'+str(i) if len(str(i))==1 else i))], axis=0)
        train_out = pd.concat([train_out, pd.read_csv(dir / 'train/output_2023_w{}.csv'.format('0'+str(i) if len(str(i))==1 else i))], axis=0)
    return train_in, train_out, test_in, test_out



In [42]:
trace_num = 10

def get_all_data(df_in, df_out):    
    ids = {}
    x = []
    y = []
    id_to_values = []
    idx = 0
    
    rows = []
    
    x_idx = list(df_in.columns).index('x')
    y_idx = list(df_in.columns).index('y')
    x_ball_idx = list(df_in.columns).index('ball_land_x')
    y_ball_idx = list(df_in.columns).index('ball_land_y')
    
    for row in df_in.itertuples(index=False, name=None):
        t = (row[0], row[1], row[3])
        if t in ids:
            j = ids[t]
        else:
            ids[t] = idx
            j = idx
            idx += 1
            id_to_values.append(row)
            x.append([])
            y.append([])
            
        if not x[j]:
            x[j].append(row[x_idx])
            y[j].append(row[y_idx])
            continue
    
        values = list(id_to_values[j])
        values[x_idx] = x[j][-1]
        values[y_idx] = y[j][-1]
        x[j].append(row[x_idx])
        y[j].append(row[y_idx])
    
 
        values.append(values[x_ball_idx]-values[x_idx])
        values.append(values[y_ball_idx]-values[y_idx])
        
        for k in range(min(trace_num, len(x[j])-1)):
            values.append(x[j][~k]-x[j][~(k+1)])
            values.append(y[j][~k]-y[j][~(k+1)])
        values += [0]*(trace_num-len(x[j])+1)*2
        rows.append(values)

    
    x_idx_out = list(df_out.columns).index('x')
    y_idx_out = list(df_out.columns).index('y')
    
    for row in df_out.itertuples(index=False, name=None):
        j = ids[row[:3]]
        values = list(id_to_values[j])
        values[x_idx] = x[j][-1]
        values[y_idx] = y[j][-1]
        values.append(values[x_ball_idx]-values[x_idx])
        values.append(values[y_ball_idx]-values[y_idx])
        x[j].append(row[x_idx_out])
        y[j].append(row[y_idx_out])

        if len(x[j])<trace_num+1:
            continue
        for k in range(min(trace_num, len(x[j])-1)):
            values.append(x[j][~k]-x[j][~(k+1)])
            values.append(y[j][~k]-y[j][~(k+1)])
        values += [0]*(trace_num-len(x[j])+1)*2
        rows.append(values)

    return rows

In [43]:
def LE_train(df_):
    df = df_.copy()
    encode_list = {}
    for col in df.select_dtypes(include=['object', 'bool']).columns:
    # for col in df.select_dtypes(exclude=['Int64', 'Float64', 'Int32', 'int64']).columns:
        unique = df[col].unique()
        encoder = {u:i for i, u in enumerate(unique)}
        df[col] = df[col].map(encoder)
        encode_list[col] = encoder
    return df, encode_list

def LE_test(df_, encode_list):
    df = df_.copy()
    for col in df.select_dtypes(include=['object', 'bool']).columns:
    # for col in df.select_dtypes(exclude=['Int64', 'Float64', 'Int32', 'int64']).columns:
        df[col] = df[col].map(encode_list[col])
        df[col] = df[col].fillna(-1).astype(int)
    return df

def f(df_in_, df_out_, encode_list=None):
    df_in = df_in_.copy()
    df_out = df_out_.copy()

    pbd = 'player_birth_date'
    if pbd in df_in.columns:
        pbd_dataframe = df_in[pbd].copy()
        df_in[pbd] = pd.to_datetime(pbd_dataframe)
        year = df_in[pbd].dt.year
        month = df_in[pbd].dt.month
        day = df_in[pbd].dt.day
        df_in[pbd] = year*400 + month*31 + day

    ph = 'player_height'
    if ph in df_in.columns:
        left_right = df_in[ph].str.split('-', expand=True)
        df_in[ph] = left_right[0].astype(int) * 12 + left_right[1].astype(int)
        
    if encode_list:
        df_in_2 = LE_test(df_in, encode_list)
    else:
        df_in_2, encode_list = LE_train(df_in)

    return df_in_2, df_out, encode_list


    

In [44]:
train_in, train_out, test_in, test_out = read()

In [45]:
train_in_1, train_out_1, encode_list = f(train_in, train_out)

In [46]:
train_in_1.head()

,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
0,2023090700,101,0,54527,1,0,42,0,73,210,799886,0,0,0,52.33,36.94,0.09,0.39,322.40,238.24,21,63.259998,-0.22
1,2023090700,101,0,54527,2,0,42,0,73,210,799886,0,0,0,52.33,36.94,0.04,0.61,200.89,236.05,21,63.259998,-0.22
2,2023090700,101,0,54527,3,0,42,0,73,210,799886,0,0,0,52.33,36.93,0.12,0.73,147.55,240.60,21,63.259998,-0.22
3,2023090700,101,0,54527,4,0,42,0,73,210,799886,0,0,0,52.35,36.92,0.23,0.81,131.40,244.25,21,63.259998,-0.22
4,2023090700,101,0,54527,5,0,42,0,73,210,799886,0,0,0,52.37,36.90,0.35,0.82,123.26,244.25,21,63.259998,-0.22


In [47]:
base = list(train_in_1.columns)

# cols = ['nfl_id', 'absolute_yardline_number',
#        'player_height', 'player_weight', 'player_birth_date',
#        'player_position', 'x', 'y',
#        'ball_land_x', 'ball_land_y']

cols = ['absolute_yardline_number',
       'player_height', 'player_weight', 'player_birth_date',
        'x', 'y',
       'ball_land_x', 'ball_land_y']



cols_add = ['dist_x', 'dist_y']

for i in range(trace_num):
    cols_add.append('dx_{}'.format(i))
    cols_add.append('dy_{}'.format(i))


# cols_test = ['game_id', 'play_id'] + cols
cols_test = ['game_id', 'play_id', 'nfl_id'] + cols
cols_add_test = cols_add[:2] + cols_add[4:]


rows_train = get_all_data(train_in_1, train_out_1)
all_data_train = pd.DataFrame(rows_train, columns=base + cols_add)

print(all_data_train.shape)

(120879, 45)


In [48]:
all_data_train.head(1)

,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y,dist_x,dist_y,dx_0,dy_0,dx_1,dy_1,dx_2,dy_2,dx_3,dy_3,dx_4,dy_4,dx_5,dy_5,dx_6,dy_6,dx_7,dy_7,dx_8,dy_8,dx_9,dy_9
0,2023090700,101,0,54527,1,0,42,0,73,210,799886,0,0,0,52.33,36.94,0.09,0.39,322.4,238.24,21,63.259998,-0.22,10.929998,-37.16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [49]:
# import lightgbm as lgb 
# from sklearn.metrics import mean_squared_error
# from itertools import combinations


# mask = np.random.rand(all_data_train.shape[0]) < 0.8

# # cols_list = []
# # scores_x = []
# # scores_y = [] 

# X = all_data_train[[col for col in cols+cols_add if col not in {'dx_0', 'dy_0'}]]
# y_x = all_data_train['dx_0']
# y_y = all_data_train['dy_0']

# # if trial:
# X_train, y_x_train, y_y_train = X[mask], y_x[mask], y_y[mask]
# X_valid, y_x_valid, y_y_valid = X[~mask], y_x[~mask], y_y[~mask]
# num_round = 1000

# evals_result_x = {}
# evals_result_y = {}

# callbacks_x = [
#     lgb.early_stopping(stopping_rounds=10),    
#     lgb.log_evaluation(100),
#     lgb.record_evaluation(evals_result_x)
# ]

# callbacks_y = [
#     lgb.early_stopping(stopping_rounds=10),    
#     lgb.log_evaluation(100),
#     lgb.record_evaluation(evals_result_y)
# ]
# # else:
# #     X_train, y_x_train, y_y_train = X, y_x, y_y
# #     X_valid, y_x_valid, y_y_valid = X.iloc[0:1], y_x.iloc[0:1], y_y.iloc[0:1]
# #     X_train, y_x_train, y_y_train = X[mask], y_x[mask], y_y[mask]
# #     X_valid, y_x_valid, y_y_valid = X[~mask], y_x[~mask], y_y[~mask]
    
# #     num_round = 1000
    
# #     evals_result_x = {}
# #     evals_result_y = {}
    
# #     callbacks_x = [lgb.log_evaluation(100),]
# #     callbacks_y = [lgb.log_evaluation(100),]
    

# lgb_train_x = lgb.Dataset(X_train, y_x_train)
# lgb_eval_x = lgb.Dataset(X_valid, y_x_valid)

# lgb_train_y = lgb.Dataset(X_train, y_y_train)
# lgb_eval_y = lgb.Dataset(X_valid, y_y_valid)

# params = {'objective': 'regression', 'seed': 71, 'verbose': 1, 
#          'metrics': 'rmse', 'learning_rate': 0.1, 'num_leaves': 31,
#          }


# model_x = lgb.train(
#     params, 
#     train_set = lgb_train_x,
#     num_boost_round = num_round,
#     valid_sets = [lgb_train_x, lgb_eval_x],
#     valid_names = ['train', 'eval'],
#     callbacks = callbacks_x,
# )

# model_y = lgb.train(
#     params, 
#     train_set = lgb_train_y,
#     num_boost_round = num_round,
#     valid_sets = [lgb_train_y, lgb_eval_y],
#     valid_names = ['train', 'eval'],
#     callbacks = callbacks_y,
# )


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

mask = np.random.rand(all_data_train.shape[0]) < 0.8


X = all_data_train[[col for col in cols+cols_add if col not in {'dx_0', 'dy_0'}]]
y = all_data_train[['dx_0', 'dy_0']]

X = X.values.astype(np.float32)
y = y.values.astype(np.float32)

X_train, y_train = X[mask], y[mask]
X_valid, y_valid = X[~mask], y[~mask]

train = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
valid = TensorDataset(torch.tensor(X_valid), torch.tensor(y_valid))

train_loader = DataLoader(train, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid, batch_size=128)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=[128]*5 + [64]*5, out_dim=1):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        layers += [nn.Linear(prev, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = MLP(in_dim=X.shape[1], out_dim=2)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(20):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    val_loss = 0
    model.eval()
    with torch. no_grad():
        for xb, yb in valid_loader:
            val_loss += loss_fn(model(xb), yb).item()
    print(f"Epoch {epoch+1:02d} | Train Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(valid_loader):.4f}")


Epoch 01 | Train Loss: 13.9030 | Val Loss: 0.1025
Epoch 02 | Train Loss: 0.2688 | Val Loss: 0.0967
Epoch 03 | Train Loss: 0.1245 | Val Loss: 0.0947
Epoch 04 | Train Loss: 0.1035 | Val Loss: 0.0943
Epoch 05 | Train Loss: 0.0985 | Val Loss: 0.0943
Epoch 06 | Train Loss: 0.0965 | Val Loss: 0.0943
Epoch 07 | Train Loss: 0.0956 | Val Loss: 0.0943
Epoch 08 | Train Loss: 0.0953 | Val Loss: 0.0943
Epoch 09 | Train Loss: 0.0951 | Val Loss: 0.0943
Epoch 10 | Train Loss: 0.0950 | Val Loss: 0.0943


In [34]:
# tmp = torch.tensor([[0]*(len(cols)+len(cols_add)-2)], dtype=torch.float32)
# model.eval()
# with torch.no_grad():
#     a, b = model(tmp)[0]
#     print(a.item(), b.item())

0.011688007041811943 -0.005092127714306116


In [37]:
def predict(df_out, df_in):
    if not isinstance(df_in, pd.DataFrame):
        df_in = df_in.to_pandas()
        df_out = df_out.to_pandas()
    df_in_1, df_out_1, _ = f(df_in, df_out, encode_list)
    df_out_1 = df_out_1.drop(columns=['id'])
    df_in_1 = df_in_1[cols_test]

    ids = {}
    x = []
    y = []
    id_to_values = []
    idx = 0
    

    num_cols = len(cols_test) + len(cols_add_test)
    x_idx = cols_test.index('x')
    y_idx = cols_test.index('y')
    x_ball_idx = cols_test.index('ball_land_x')
    y_ball_idx = cols_test.index('ball_land_y')

    predictions = [[], []]
    
    for row in df_in_1.itertuples(index=False, name=None):
        t = row[:3]
        if t in ids:
            j = ids[t]
        else:
            ids[t] = idx
            j = idx
            idx += 1
            id_to_values.append(row)
            x.append([])
            y.append([])
            
        x[j].append(row[x_idx])
        y[j].append(row[y_idx])


    # tmp = []
    for row in df_out_1.itertuples(index=False, name=None):
        t = row[:3]
        j = ids[t]
        values = list(id_to_values[j])
        values[x_idx] = x[j][-1]
        values[y_idx] = y[j][-1]
        values.append(values[x_ball_idx]-values[x_idx])
        values.append(values[y_ball_idx]-values[y_idx])
        for k in range(min(trace_num-1, len(x[j])-1)):
            values.append(x[j][~k]-x[j][~(k+1)])
            values.append(y[j][~k]-y[j][~(k+1)])
        values += [0]*(trace_num-len(x[j]))*2

        values = torch.tensor([values[3:]], dtype=torch.float32)
        model.eval()
        with torch.no_grad():
            dx, dy = model(values)[0]

        nx, ny = x[j][-1]+dx.item(), y[j][-1]+dy.item()

        x[j].append(nx)
        y[j].append(ny)
        predictions[0].append(nx)
        predictions[1].append(ny)

        

    submission = pd.DataFrame({'x': predictions[0], 'y': predictions[1]})
    # print(tmp)
    return submission
    

In [38]:
# test_in= pd.read_csv(dir / 'test_input.csv')
# test_out = pd.read_csv(dir / 'test.csv')
# preds = predict(test_out, test_in)
# preds

,x,y
0,88.071954,34.311628
1,88.083908,34.303257
2,88.095862,34.294885
3,88.107815,34.286514
4,88.119769,34.278142
...,...,...
5832,81.110800,16.182340
5833,81.122754,16.173968
5834,81.134708,16.165596
5835,81.146662,16.157225


In [39]:
import kaggle_evaluation.nfl_inference_server
import os
inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/nfl-big-data-bowl-2026-prediction/',))
